In [ ]:
import os
import pandas as pd
import geopandas as gpd
import geobr

# Importar as funções desenvolvidas nos módulos da pasta src/soja_clima/
from src.soja_clima.qualidade import filtrar_estacoes_fora_do_brasil
from src.soja_clima.sidra import ler_sidra_soja, agregar_soja_para_microrregiao
from src.soja_clima.geo import construir_mapa_estacao_microrregiao
from src.soja_clima.agregacao import (
    agregar_diario_para_estacao_safra,
    agregar_espacial_para_microrregiao,
)
from src.soja_clima.merge import (
    montar_base_final,
    # [ACRÉSCIMO] etapas vindas da refatoração (antigos 05_merge_final / correcoes_filtros)
    LIMITES_FISICOS,
    agregar_clima_micro_safra,
    agregar_pacifico_safra,
    adicionar_solo,
    selecionar_colunas_solo,
    filtrar_base_modelagem,
    calcular_desvio_clima,
)

In [ ]:
# ==============================================================================
# 1. CONFIGURAÇÃO DOS CAMINHOS
# ==============================================================================
# Raiz do projeto: o notebook fica na raiz, então é o diretório de trabalho.
# Pode ser sobrescrito pela variável de ambiente PCD2_BASE_DIR.
BASE_DIR = os.environ.get("PCD2_BASE_DIR", os.getcwd())

CAMINHO_CLIMA = os.path.join(BASE_DIR, "data/processed/clima_diario.parquet")
CAMINHO_SIDRA = os.path.join(BASE_DIR, "data/raw/tabela1612.csv")
# [ACRÉSCIMO] entradas geradas por notebooks/pipeline/03 (solo) e 04 (Pacífico)
CAMINHO_SOLO = os.path.join(BASE_DIR, "data/processed/solo_microrregioes_limpo.parquet")
CAMINHO_PACIFICO = os.path.join(BASE_DIR, "data/processed/pacifico_indices.parquet")

# Saída ÚNICA do projeto
CAMINHO_SAIDA = os.path.join(BASE_DIR, "data/processed/base_final_modelagem.csv")

print("--- A iniciar o Pipeline de Integração ---")


In [ ]:
# ==============================================================================
# 2. CARREGAMENTO E QUALIDADE DOS DADOS CLIMÁTICOS
# ==============================================================================
print("\n[1/8] A carregar e a filtrar dados climáticos...")
diario = pd.read_parquet(CAMINHO_CLIMA)
diario["data"] = pd.to_datetime(diario["data"])

# O clima_diario.parquet do pipeline da refatoração usa VL_LATITUDE/VL_LONGITUDE;
# o fluxo da main (qualidade/geo) espera latitude/longitude.
diario = diario.rename(columns={"VL_LATITUDE": "latitude", "VL_LONGITUDE": "longitude"})

# Filtrar estações localizadas fora da caixa geográfica do Brasil
diario, estacoes_removidas = filtrar_estacoes_fora_do_brasil(diario)
# [ACRÉSCIMO] Limpeza física dos sensores (Alerta 6 da EDA)
antes = len(diario)
diario = diario[
    (diario["temp_min"] >= LIMITES_FISICOS["temp_min_min"])
    & (diario["prec_total"] <= LIMITES_FISICOS["prec_total_max"])
    & (diario["gdd"] <= LIMITES_FISICOS["gdd_max"])
].copy()
print(f"   - {antes - len(diario):,} dias removidos pelos limites físicos")
print(f"   ✓ Dados climáticos prontos. Registos válidos: {len(diario):,}")


In [ ]:
# ==============================================================================
# 3. LEITURA E PREPARAÇÃO DOS DADOS DE PRODUÇÃO DE SOJA (SIDRA)
# ==============================================================================
print("\n[2/8] A ler e a processar dados da Tabela 1612 do SIDRA...")
df_soja_mun = ler_sidra_soja(CAMINHO_SIDRA)

print("   - Descarregando malhas do geobr e associando municípios às microrregiões...")
mun_br = geobr.read_municipality(year=2020)
microrregioes = geobr.read_micro_region(year=2020)

# Join espacial: usa o centroide de cada município para identificar sua microrregião
mun_br_centroids = mun_br.copy()
mun_br_centroids["geometry"] = mun_br_centroids.geometry.centroid

mun_com_micro = gpd.sjoin(
    mun_br_centroids,
    microrregioes[["code_micro", "name_micro", "geometry"]],
    how="left",
    predicate="within"
)

# Normalizar texto dos municípios para garantir o correspondência exata no merge
df_soja_mun["mun_join"] = (
    df_soja_mun["municipio"]
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)
mun_com_micro["mun_join"] = (
    mun_com_micro["name_muni"]
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
)

# Associar o código da microrregião (code_micro) aos dados municipais do SIDRA
df_soja_mun_com_micro = df_soja_mun.merge(
    mun_com_micro[["mun_join", "abbrev_state", "code_micro", "name_micro"]],
    left_on=["mun_join", "uf"],
    right_on=["mun_join", "abbrev_state"],
    how="inner",
).drop(columns=["mun_join", "abbrev_state"])

# Agregar a produção agrícola ao nível de microrregião e recalcular o rendimento (kg/ha)
df_soja_micro = agregar_soja_para_microrregiao(df_soja_mun_com_micro)
print(f"   ✓ Produção agregada por microrregião. Linhas: {len(df_soja_micro):,}")

In [ ]:
# ==============================================================================
# 4. MAPEAMENTO ESPACIAL (ESTAÇÕES METEOROLÓGICAS -> MICRORREGIÕES)
# ==============================================================================
print("\n[3/8] A descarregar malha de microrregiões e a efetuar o join espacial...")
microrregioes = geobr.read_micro_region(year=2020)

# Mapear cada estação à respetiva microrregião (com tratamento para pontos fora de polígonos)
mapa_estacoes = construir_mapa_estacao_microrregiao(diario, microrregioes)

# Incorporar o 'code_micro' ao dataframe diário do clima
diario_com_micro = diario.merge(
    mapa_estacoes[["codigo_estacao", "code_micro"]],
    on="codigo_estacao",
    how="inner",
)
print("   ✓ Join espacial das estações concluído.")

In [ ]:
# ==============================================================================
# 5. AGREGAÇÃO TEMPORAL E ESPACIAL DO CLIMA
# ==============================================================================
print("\n[4/8] A agregar clima no tempo (Diário -> Estação/Safra por fases fenológicas)...")
estacao_safra = agregar_diario_para_estacao_safra(diario_com_micro, usar_fases=True)

print("\n[5/8] A agregar clima no espaço (Estação/Safra -> Microrregião/Safra)...")
clima_micro_safra = agregar_espacial_para_microrregiao(estacao_safra)
print(f"   ✓ Clima agregado por microrregião e safra (fases). Linhas: {len(clima_micro_safra):,}")

# [ACRÉSCIMO] Totais da safra inteira (mesma lógica: soma por estação, média entre estações)
print("   - Totais da safra inteira + índices do Pacífico...")
clima_safra_total = agregar_clima_micro_safra(diario, mapa_estacoes, aplicar_limites_fisicos=False)
pacifico_safra = agregar_pacifico_safra(pd.read_parquet(CAMINHO_PACIFICO))

In [ ]:
# ==============================================================================
# 6. MERGE FINAL E EXPORTAÇÃO
# ==============================================================================
print("\n[6/8] A realizar o merge final entre Clima e Produção de Soja...")
base_final = montar_base_final(clima_micro_safra, df_soja_micro)

# [ACRÉSCIMO] Totais da safra, Pacífico (por ano-safra) e solo (estático)
base_final = adicionar_solo(base_final, pd.read_parquet(CAMINHO_SOLO))  # cria CD_MICRO (string)
nomes_micro = pd.DataFrame({"code_micro": microrregioes["code_micro"], "NM_MICRO": microrregioes["name_micro"]})
base_final = base_final.merge(nomes_micro, on="code_micro", how="left")
# n_estacoes já vem da agregação por fases (mesmas estações) -> não duplicar
base_final = base_final.merge(clima_safra_total.drop(columns=["n_estacoes"]), on=["CD_MICRO", "ano"], how="left")
base_final = base_final.merge(pacifico_safra, on="ano", how="left")
assert not base_final.duplicated(["code_micro", "ano"]).any(), "duplicatas em (microrregião, ano)"
print(f"   ✓ Base integrada: {base_final.shape}")


In [ ]:
# ==============================================================================
# 7. [ACRÉSCIMO] FILTROS, SELEÇÃO DE SOLO E DETRENDING (antigo correcoes_filtros)
# ==============================================================================
print("\n[7/8] A aplicar filtros da EDA e a remover a tendência tecnológica...")
base_final = filtrar_base_modelagem(base_final, area_min_ha=100, rendimento_max_kg_ha=6000, ano_min=2009)
base_final = selecionar_colunas_solo(base_final)
base_final = calcular_desvio_clima(base_final)
print(f"   ✓ Base de modelagem: {base_final.shape} | anos {base_final['ano'].min()}–{base_final['ano'].max()}")


In [ ]:
# ==============================================================================
# 8. EXPORTAÇÃO (saída única do projeto)
# ==============================================================================
print("\n[8/8] A exportar a base final...")
os.makedirs(os.path.dirname(CAMINHO_SAIDA), exist_ok=True)
base_final.to_csv(CAMINHO_SAIDA, index=False)

print("\n==================================================================")
print("--- Pipeline concluído com sucesso! ---")
print(f"Ficheiro guardado em: {CAMINHO_SAIDA}")
print(f"Dimensão da base final pronta para modelagem: {base_final.shape}")
print("==================================================================")
